In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

project_root = Path.cwd().parent
sys.path.append(str(project_root))

from src.alpha_correlations import void_fraction_correlations
from src.combined_properties import properties_df
from src.void_fraction_per_pattern import void_fraction_calculation_function

In [3]:
pd.set_option('display.max_rows', 100)

CORR_COLS = [
    'alpha_h', 'lockhart', 'zivi', 'premoli', 'kanizawa',
    'tibirica', 'hughmark', 'rouhani', 'steiner',
]
T2_MISSING_MAX = 50

# Load data

In [4]:
silver_path = project_root / 'data' / 'silver_layer'
input_path = silver_path / 'confocal_runs'
output_path = project_root / 'data' / 'gold_layer' / 'confocal_void_fraction'
output_path.mkdir(parents=True, exist_ok=True)

df_confocal = pd.read_pickle(silver_path / 'confocal_results_processed.pkl')
df_experimental = pd.read_pickle(silver_path / 'experimental_results.pkl')

# Confocal void fraction (per run)

In [5]:
vf_results = []

for file in sorted(input_path.glob('confocal_run_*.csv')):
    match = re.search(r'confocal_run_(\d+)', file.stem)
    if not match:
        continue

    run_id = int(match.group(1))
    df_run = pd.read_csv(file)

    reduced_pattern = (
        df_experimental.loc[
            df_experimental['run_id'] == run_id,
            'reduced_pattern',
        ].iloc[0]
    )

    void_fraction = void_fraction_calculation_function(df_run, reduced_pattern)

    vf_results.append({
        'run_id': run_id,
        'void_fraction': void_fraction,
    })

df_vf = pd.DataFrame(vf_results).sort_values('run_id').reset_index(drop=True)

C:\Users\fernando\Downloads\projects\phd_paper\src\void_fraction_per_pattern.py:88: RuntimeWarning: Mean of empty slice
  return np.nanmean(vf)
C:\Users\fernando\Downloads\projects\phd_paper\src\void_fraction_per_pattern.py:88: RuntimeWarning: Mean of empty slice
  return np.nanmean(vf)


# Thermophysical properties and void-fraction correlations

In [6]:
df_consolidated = pd.merge(df_experimental, df_vf, on='run_id', how='inner')

df_properties = properties_df(df_consolidated)
df_correlations = void_fraction_correlations(df_properties)

df_intermediate = pd.merge(
    df_consolidated,
    df_correlations[['run_id'] + CORR_COLS],
    on='run_id',
    how='inner',
)

df = pd.merge(
    df_intermediate,
    df_confocal[['run_id', 't2_missing_percentage']],
    on='run_id',
    how='inner',
)

df = df[df['t2_missing_percentage'] < T2_MISSING_MAX].reset_index(drop=True)


# Comparison: experimental vs correlations

In [7]:
id_cols = ['run_id', 'reduced_pattern', 'void_fraction']

df_wide = df[id_cols + CORR_COLS].copy()
for col in CORR_COLS:
    df_wide[f'error_{col}'] = df_wide[col] - df_wide['void_fraction']

error_cols = [f'error_{col}' for col in CORR_COLS]
df_wide = df_wide[id_cols + CORR_COLS + error_cols]

df_comparison = df_wide.melt(
    id_vars=id_cols,
    value_vars=CORR_COLS,
    var_name='correlation',
    value_name='void_fraction_corr',
)
_errors = df_wide.melt(
    id_vars=id_cols,
    value_vars=error_cols,
    var_name='correlation',
    value_name='error',
)
_errors['correlation'] = _errors['correlation'].str.removeprefix('error_')

df_comparison = df_comparison.merge(_errors, on=id_cols + ['correlation'])
df_comparison['abs_error'] = df_comparison['error'].abs()

In [8]:
summary = (
    df_comparison
    .groupby(['reduced_pattern', 'correlation'], observed=True)
    .agg(
        n_runs=('run_id', 'count'),
        mae=('abs_error', 'mean'),
        rmse=('error', lambda s: np.sqrt((s ** 2).mean())),
        bias=('error', 'mean'),
    )
    .reset_index()
    .sort_values(['reduced_pattern', 'mae'])
)

# Save

In [9]:
save_path = project_root / 'data' / 'gold_layer'

df_comparison.to_csv(
    save_path / 'void_fraction_results.csv',
)

df_comparison.to_pickle(
    save_path / 'void_fraction_results.pkl',
)

summary.to_csv(
    save_path / 'void_fraction_grouped_results.csv'
)

summary.to_pickle(
    save_path / 'void_fraction_grouped_results.pkl'
)

In [10]:
df_comparison.query('reduced_pattern == "wavy"').sort_values('run_id')

,run_id,reduced_pattern,void_fraction,correlation,void_fraction_corr,error,abs_error
14,40,wavy,0.866364,alpha_h,0.989362,0.122998,0.122998
83,40,wavy,0.866364,lockhart,0.877227,0.010863,0.010863
221,40,wavy,0.866364,premoli,0.798807,-0.067557,0.067557
152,40,wavy,0.866364,zivi,0.905110,0.038746,0.038746
428,40,wavy,0.866364,hughmark,0.809289,-0.057075,0.057075
497,40,wavy,0.866364,rouhani,0.851595,-0.014769,0.014769
359,40,wavy,0.866364,tibirica,0.913269,0.046905,0.046905
290,40,wavy,0.866364,kanizawa,0.844483,-0.021881,0.021881
566,40,wavy,0.866364,steiner,0.845007,-0.021357,0.021357
153,41,wavy,0.855995,zivi,0.798786,-0.057209,0.057209


In [11]:
summary

,reduced_pattern,correlation,n_runs,mae,rmse,bias
4,bubbly,premoli,9,0.064696,0.088006,0.057018
3,bubbly,lockhart,9,0.069222,0.088459,0.059448
8,bubbly,zivi,9,0.070684,0.091505,-0.064206
1,bubbly,hughmark,9,0.076520,0.106676,0.076520
6,bubbly,steiner,9,0.087207,0.114602,0.087207
5,bubbly,rouhani,9,0.090212,0.117178,0.090212
2,bubbly,kanizawa,9,0.101890,0.118567,0.101890
0,bubbly,alpha_h,9,0.129586,0.155634,0.129586
7,bubbly,tibirica,9,0.158245,0.171633,0.158245
16,dispersed,tibirica,11,0.248316,0.301401,0.012976


In [12]:
round(summary.groupby('correlation')[['mae',	'rmse', 'bias']].mean(), 3)

,mae,rmse,bias
correlation,,,
alpha_h,0.222,0.264,0.184
hughmark,0.157,0.199,0.059
kanizawa,0.143,0.183,0.036
lockhart,0.144,0.181,-0.008
premoli,0.146,0.184,-0.004
rouhani,0.154,0.197,0.088
steiner,0.153,0.196,0.081
tibirica,0.170,0.208,0.119
zivi,0.192,0.221,-0.149
